1. IMPORTY

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, confusion_matrix

2. WCZYTANIE DANYCH

In [2]:
# baza danych: Wine Quality from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

print("--- Pierwsze 5 wierszy ---")
print(df.head())

--- Pierwsze 5 wierszy ---
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8     

3. EDA - PODSTAWOWY PRZEGLĄD

In [3]:
print(f"\nKształt danych: {df.shape}")
df.info()
print(df.describe(include='all'))
print("\nBrakujące wartości:")
print(df.isna().sum())


Kształt danych: (1599, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB
       fixed acidity  volatile acidity  citric acid  residual sugar  \
count    1599.000000       

4. ROZKŁADY I OUTLIERY

In [4]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
plt.figure()
df[col].hist(bins=30)
plt.title(f'Histogram: {col}')
plt.show()

for col in numeric_cols:
plt.figure()
df.boxplot(column=col)
plt.title(f'Boxplot: {col}')
plt.show()

IndentationError: expected an indented block after 'for' statement on line 3 (380287461.py, line 4)

5. KORELACJE

In [5]:
corr = df.corr(numeric_only=True)
print("\nMacierz korelacji:")
print(corr)


Macierz korelacji:
                      fixed acidity  volatile acidity  citric acid  \
fixed acidity              1.000000         -0.256131     0.671703   
volatile acidity          -0.256131          1.000000    -0.552496   
citric acid                0.671703         -0.552496     1.000000   
residual sugar             0.114777          0.001918     0.143577   
chlorides                  0.093705          0.061298     0.203823   
free sulfur dioxide       -0.153794         -0.010504    -0.060978   
total sulfur dioxide      -0.113181          0.076470     0.035533   
density                    0.668047          0.022026     0.364947   
pH                        -0.682978          0.234937    -0.541904   
sulphates                  0.183006         -0.260987     0.312770   
alcohol                   -0.061668         -0.202288     0.109903   
quality                    0.124052         -0.390558     0.226373   

                      residual sugar  chlorides  free sulfur dioxide 

6. MODEL REGRESYJNY

In [11]:
#Kodowanie zmiennych:
data=df.copy()
data_encoded = pd.get_dummies(data, drop_first=True)
X = data_encoded.drop('quality', axis=1)
y = data_encoded['quality']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
y_pred = reg_model.predict(X_test)

print("\n--- Regresja ---")
print(f"MSE: {mean_squared_error(y_test, y_pred)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred)}")



--- Regresja ---
MSE: 0.3900251439639549
MAE: 0.5035304415524375


7.MODEL KLASYFIKACYJNY

In [12]:
y_bin = (y > 5).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
X_scaled, y_bin, test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_c, y_train_c)
y_pred_c = clf.predict(X_test_c)

print("\n--- Klasyfikacja ---")
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_c)}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_c, y_pred_c))


--- Klasyfikacja ---
Accuracy: 0.740625
Confusion Matrix:
[[105  36]
 [ 47 132]]


Autor: Kamil Pilszczek